In [ ]:
import os
import re
import json
import random
import ast
from collections import defaultdict

import numpy as np
import torch
from tqdm.auto import tqdm
from PIL import Image

import pandas as pd
from datasets import load_dataset, get_dataset_config_names, concatenate_datasets

from transformers import (
    AutoModelForImageTextToText,
    AutoProcessor,
)

from IPython.display import display  # for image visualization

# --------------------------
# Config
# --------------------------
BASE_MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
FT_CKPT = "/work/wildfirerisk/trainings/small/oracles/qwen_vlm_grpo/checkpoint-1800"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

MAX_NEW_TOKENS = 1024
SEED = 42
BATCH_SIZE = 16          # or whatever fits in VRAM
NUM_SAMPLES_LIMIT = -1   # -1 = use all validation examples
RUN_DEBUG = False       # True = single-example debug; False = full eval
VERBOSE_EVAL_DEBUG = False  # set True to print per-question outputs

# --------------------------
# Determinism
# --------------------------
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# --------------------------
# MMMU loading utilities
# --------------------------
def load_mmmu_validation():
    """
    Load all validation splits across all MMMU subjects and concatenate.
    """
    dataset_name = "MMMU/MMMU"
    subjects = get_dataset_config_names(dataset_name)
    val_dsets = []
    print(f"Found {len(subjects)} MMMU subjects. Loading validation splits...")

    for subject in tqdm(subjects):
        d = load_dataset(
            dataset_name,
            subject,
            split="validation",
        )
        val_dsets.append(d)

    full_val = concatenate_datasets(val_dsets)
    print(f"Total MMMU validation examples: {len(full_val)}")
    return full_val


def get_first_image(row):
    """
    MMMU has up to 7 image fields: image_1 ... image_7.
    For the model, we use only the first non-null image.
    """
    for i in range(1, 8):
        key = f"image_{i}"
        if key in row and row[key] is not None:
            img = row[key]
            if isinstance(img, Image.Image):
                return img.convert("RGB")
    # Fallback: blank placeholder (should be rare)
    return Image.new("RGB", (256, 256), color="white")


def visualize_all_images(row):
    """
    Display all available images in the example for debugging.
    """
    imgs = []
    for i in range(1, 8):
        key = f"image_{i}"
        if key in row and row[key] is not None and isinstance(row[key], Image.Image):
            imgs.append(row[key].convert("RGB"))

    if not imgs:
        print("No images found for this example.")
        return

    for idx, img in enumerate(imgs, start=1):
        print(f"Image {idx}:")
        display(img)

# --------------------------
# Prompt builder
# --------------------------
LETTER_TO_INDEX = {chr(ord("A") + i): i for i in range(10)}
INDEX_TO_LETTER = {v: k for k, v in LETTER_TO_INDEX.items()}

FINAL_RE = re.compile(r"FINAL ANSWER\s*:\s*([A-J])", re.IGNORECASE)


def clean_question_text(q: str) -> str:
    q = re.sub(r"<image\s*\d+>", "[IMAGE]", q)
    return q.strip()


def normalize_options(raw_options):
    """
    MMMU/MMMU often stores 'options' as a string that *looks* like a Python list.
    Normalize to a real Python list of strings.
    """

    # Already a list
    if isinstance(raw_options, list):
        return [str(x) for x in raw_options]

    # String: try literal_eval
    if isinstance(raw_options, str):
        raw_options_str = raw_options.strip()
        try:
            parsed = ast.literal_eval(raw_options_str)
            if isinstance(parsed, list):
                return [str(x) for x in parsed]
        except Exception as e:
            print("ast.literal_eval failed on 'options':", e)

        # Fallback 1: split on newlines
        if "\n" in raw_options_str:
            parts = [p.strip() for p in raw_options_str.split("\n") if p.strip()]
            if len(parts) > 1:
                return parts

        # Fallback 2: treat as a single option
        return [raw_options_str]

    # Unexpected type: fallback
    return [str(raw_options)]


def build_mmmu_prompt(question: str, options, subject: str = None) -> str:
    """
    Build a chain-of-thought style prompt that ends with 'FINAL ANSWER: X'.
    """
    q_clean = clean_question_text(question)
    opt_lines = []
    for i, opt in enumerate(options):
        letter = chr(ord("A") + i)
        opt_lines.append(f"{letter}. {opt}")
    opts_block = "\n".join(opt_lines)

    subject_hint = f"Subject area: {subject}.\n\n" if subject is not None else ""

    user_text = (
        "You are an expert AI assistant solving multiple-choice questions.\n"
        "You see a question, one or more images, and several answer options.\n"
        "Carefully read the question and analyze the image(s).\n"
        "Then, reason step by step and choose the single best option.\n\n"
        "Question:\n"
        f"{q_clean}\n\n"
        "Options:\n"
        f"{opts_block}\n\n"
        "First, think through the problem. Then, on the last line, output:\n"
        "FINAL ANSWER: X\n"
        "where X is the letter (A, B, C, or D, etc.) of the correct option."
    )
    return user_text


def build_conversation_for_qwen(user_text: str):
    """
    Build a Qwen-style single-turn conversation with one image placeholder.
    """
    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": user_text},
            ],
        },
    ]
    return conversation

# --------------------------
# Answer parsing
# --------------------------
def extract_answer_letter(text: str):
    """
    Extract answer letter from model output (completion *only*).
    Allow any letter A–J, and let scoring decide correctness vs GT.
    """
    text = text.strip()
    valid_letters = [chr(ord("A") + i) for i in range(10)]  # A-J

    # 1) Look for explicit 'FINAL ANSWER: X'
    m = FINAL_RE.search(text)
    if m:
        cand = m.group(1).upper()
        if cand in valid_letters:
            return cand

    # 2) Look for a line that is just a single valid letter
    for line in reversed(text.splitlines()):
        cand = line.strip().rstrip(".")
        if cand in valid_letters:
            return cand

    # 3) Last occurrence of a valid letter anywhere in the completion
    candidates = [ch for ch in text if ch in valid_letters]
    if candidates:
        return candidates[-1]

    return None

# --------------------------
# Single-example debug evaluation
# --------------------------
def run_single_debug_example(model, processor, row, tag: str):
    """
    Run a single MMMU example through the model, print prompt + full output,
    and return the parsed prediction.
    """
    model.eval()
    device = next(model.parameters()).device

    qid = row["id"]
    subject = qid.split("_")[1] if "_" in qid else "Unknown"
    question = row["question"]
    raw_options = row["options"]
    options = normalize_options(raw_options)
    answer_letter = str(row["answer"]).strip().upper()

    print(f"\n===== [{tag}] Example ID: {qid} =====")
    print(f"Subject: {subject}")
    print("\nQuestion text:")
    print(question)
    print("\nNormalized options:")
    for i, opt in enumerate(options):
        print(f"{chr(ord('A') + i)}. {opt}")
    print(f"\nGround-truth answer: {answer_letter}")

    # Visualize images
    print("\nVisualizing all images for this example...")
    visualize_all_images(row)

    # Build prompt
    user_text = build_mmmu_prompt(question, options, subject=subject)
    conv = build_conversation_for_qwen(user_text)
    text_prompt = processor.apply_chat_template(
        conv,
        add_generation_prompt=True,
        tokenize=False,
    )

    print("\n------- Chat Prompt (text) -------")
    print(text_prompt)

    # For the model we only pass the first image
    img = get_first_image(row)

    inputs = processor(
        text=[text_prompt],
        images=[img],
        padding=True,
        return_tensors="pt",
    ).to(device)

    with torch.inference_mode():
        gen_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
        )

    # Trim off prompt tokens – decode only completion
    input_ids = inputs.input_ids[0]
    output_ids = gen_ids[0][len(input_ids):]

    output_text = processor.decode(
        output_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    print("\n------- Full model output -------")
    print(output_text)
    print("\n------- Output (repr, to see escapes) -------")
    print(repr(output_text))

    pred_letter = extract_answer_letter(output_text)
    print(f"\nParsed predicted answer [{tag}]: {pred_letter}")
    print(f"Correct? {pred_letter == answer_letter}")

    return {
        "id": qid,
        "subject": subject,
        "answer_letter": answer_letter,
        f"{tag}_pred_letter": pred_letter,
        f"{tag}_raw_output": output_text,
    }

# --------------------------
# Batched evaluation
# --------------------------
def evaluate_model(model, processor, dataset, tag: str, indices=None):
    """
    Evaluate one model on MMMU, returning a dict:
    results[id] = {
        "id": ...,
        "subject": ...,
        "answer_letter": ...,
        f"{tag}_pred_letter": ...,
        f"{tag}_raw_output": ...,
    }
    """
    model.eval()
    device = next(model.parameters()).device

    results = {}

    # Use provided indices or default to all
    if indices is None:
        indices = list(range(len(dataset)))

    print(f"Preparing ground truth for {len(indices)} examples...")
    for i in indices:
        row = dataset[i]
        qid = row["id"]
        subject = qid.split("_")[1] if "_" in qid else "Unknown"
        gt_letter = str(row["answer"]).strip().upper()
        results[qid] = {
            "id": qid,
            "subject": subject,
            "answer_letter": gt_letter,
        }

    print(f"Evaluating {tag} on {len(indices)} examples...")

    for start in tqdm(range(0, len(indices), BATCH_SIZE)):
        batch_idx = indices[start : start + BATCH_SIZE]
        pil_images = []
        prompt_texts = []
        qids = []
        questions = []
        options_list = []

        for idx in batch_idx:
            row = dataset[idx]
            qid = row["id"]
            question = row["question"]
            raw_options = row["options"]
            options = normalize_options(raw_options)
            subject = results[qid]["subject"]

            img = get_first_image(row)
            pil_images.append(img)

            user_text = build_mmmu_prompt(question, options, subject=subject)
            conv = build_conversation_for_qwen(user_text)
            text_prompt = processor.apply_chat_template(
                conv, add_generation_prompt=True, tokenize=False
            )
            prompt_texts.append(text_prompt)
            qids.append(qid)
            questions.append(question)
            options_list.append(options)

        # Tokenize + preprocess
        inputs = processor(
            text=prompt_texts,
            images=pil_images,
            padding=True,
            return_tensors="pt",
        )
        inputs = inputs.to(device)

        with torch.inference_mode():
            gen_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
            )

        # Trim off prompt tokens so we decode *only* the generated completion
        generated_ids_trimmed = [
            out_ids[len(in_ids):]
            for in_ids, out_ids in zip(inputs.input_ids, gen_ids)
        ]
        texts = processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )

        for qid, out_text, q_text, opts in zip(
            qids, texts, questions, options_list
        ):
            pred_letter = extract_answer_letter(out_text)
            results[qid][f"{tag}_pred_letter"] = pred_letter
            results[qid][f"{tag}_raw_output"] = out_text

            if VERBOSE_EVAL_DEBUG:
                print(f"\n==============================")
                print(f"[{tag}] qid: {qid}")
                print(f"Subject: {results[qid]['subject']}")
                print("------------------------------")
                print("Question:")
                print(q_text)
                print("\nOptions:")
                for i, opt in enumerate(opts):
                    print(f"{chr(ord('A') + i)}. {opt}")
                print(f"\nGround-truth answer: {results[qid]['answer_letter']}")
                print("\nModel raw output:")
                print(out_text)
                print("\nParsed predicted letter:", pred_letter)
                print("==============================")

    return results


# --------------------------
# Main
# --------------------------
def main():
    mmmu_val = load_mmmu_validation()

    # Choose the indices ONCE, shared across base & ft
    all_indices = list(range(len(mmmu_val)))
    if NUM_SAMPLES_LIMIT > 0 and NUM_SAMPLES_LIMIT < len(all_indices):
        random.shuffle(all_indices)
        all_indices = all_indices[:NUM_SAMPLES_LIMIT]
    print(f"\nUsing {len(all_indices)} examples total for evaluation.\n")

    print("\nLoading processor...")
    processor = AutoProcessor.from_pretrained(
        BASE_MODEL_ID,
        padding_side="left",
        use_fast=True,
    )

    print("\nLoading BASE model:", BASE_MODEL_ID)
    base_model = AutoModelForImageTextToText.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=DTYPE,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    ).to(DEVICE)

    print("\nLoading FINE-TUNED model from:", FT_CKPT)
    ft_model = AutoModelForImageTextToText.from_pretrained(
        FT_CKPT,
        torch_dtype=DTYPE,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    ).to(DEVICE)

    if RUN_DEBUG:
        # ---------- single-example debug path ----------
        idx = random.choice(all_indices)
        print(f"\n===============================")
        print(f"DEBUG: single random example: index {idx}")
        print(f"===============================\n")

        row = mmmu_val[idx]
        print("Row id:", row["id"])

        base_result = run_single_debug_example(base_model, processor, row, tag="base")
        ft_result = run_single_debug_example(ft_model, processor, row, tag="ft")

        merged = {**base_result, **ft_result}
        print("\n======================")
        print("Summary for this example")
        print("======================")
        print(json.dumps(merged, indent=2))
        return merged

    # ---------- full evaluation path ----------
    base_results = evaluate_model(
        base_model, processor, mmmu_val, tag="base", indices=all_indices
    )
    ft_results = evaluate_model(
        ft_model, processor, mmmu_val, tag="ft", indices=all_indices
    )

    # Merge result dicts
    merged = {}
    for k, v in base_results.items():
        merged[k] = dict(v)
    for k, v in ft_results.items():
        if k not in merged:
            merged[k] = {}
        for kk, vv in v.items():
            if kk not in merged[k]:
                merged[k][kk] = vv

    df = pd.DataFrame.from_dict(merged, orient="index").reset_index(drop=True)

    # Restrict df to the subset we actually evaluated
    eval_ids = set(mmmu_val[i]["id"] for i in all_indices)
    df = df[df["id"].isin(eval_ids)]

    # Keep only rows where the GT answer is a letter (A–J)
    valid_letters = set(LETTER_TO_INDEX.keys())
    df = df[df["answer_letter"].isin(valid_letters)]

    print("\nAfter filtering to letter answers and eval subset:")
    print("Total rows:", len(df))
    print("Rows with missing base_pred_letter:", df["base_pred_letter"].isna().sum())
    print("Rows with missing ft_pred_letter:", df["ft_pred_letter"].isna().sum())

    # Treat missing/None predictions as wrong
    df["base_correct"] = df["base_pred_letter"] == df["answer_letter"]
    df["ft_correct"]   = df["ft_pred_letter"]   == df["answer_letter"]

    # Overall accuracies
    base_acc = df["base_correct"].mean()
    ft_acc   = df["ft_correct"].mean()

    print("\n======================")
    print("Overall MMMU (validation) accuracy")
    print("======================")
    print(f"Base model  ({BASE_MODEL_ID}): {base_acc * 100:.2f}%")
    print(f"Fine-tuned  ({FT_CKPT}):      {ft_acc * 100:.2f}%")

    # Per-subject accuracies
    grouped = df.groupby("subject")[["base_correct", "ft_correct"]].mean().sort_index()

    print("\n======================")
    print("Per-subject accuracy (fraction correct)")
    print("======================")
    print(grouped)

    # Simple comparison stats
    improved = ((df["ft_correct"] == True) & (df["base_correct"] == False)).mean()
    regressed = ((df["ft_correct"] == False) & (df["base_correct"] == True)).mean()

    print("\n======================")
    print("Change breakdown (as fraction of evaluated questions)")
    print("======================")
    print(f"Improved (FT correct, base wrong):      {improved * 100:.2f}%")
    print(f"Regressed (FT wrong, base correct):     {regressed * 100:.2f}%")

    # Save raw outputs for later analysis
    out_path = "/work/wildfirerisk/mmmu_qwen2_5_vl_base_vs_ft_results_fixed.json"
    with open(out_path, "w") as f:
        json.dump(merged, f, indent=2)
    print(f"\nSaved detailed results (including raw model outputs) to: {os.path.abspath(out_path)}")

    # Show a few sample rows for sanity check
    print("\nSample rows:")
    display_cols = [
        "id", "subject",
        "answer_letter", "base_pred_letter", "ft_pred_letter",
        "base_correct", "ft_correct"
    ]
    display(df[display_cols].head(10))

    return df, grouped


if __name__ == "__main__":
    _ = main()
